## Domain Stopwords

In [4]:
from nltk.corpus import stopwords

# Mengambil set stopword bahasa Inggris dari NLTK
stopwords_nltk = sorted(list(stopwords.words('english')))

print(f"Total Stopwords NLTK: {len(stopwords_nltk)} kata\n")
print("Daftar Stopwords NLTK:")
print(stopwords_nltk)

Total Stopwords NLTK: 198 kata

Daftar Stopwords NLTK:
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", "he's", 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'i', "i'd", "i'll", "i'm", "i've", 'if', 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', '

In [ ]:
"""
Keunggulan TF untuk Kasus Ini: TF mengukur volume kemunculan kata secara langsung tanpa memberikan penalti (seperti IDF). 
Dengan mengurutkan nilai TF dari yang tertinggi, bisa dapat langsung melihat kata-kata bervolume tinggi yang paling mendominasi, 
sehingga sangat tepat dijadikan sebagai kandidat domain stopwords.
"""

In [6]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.corpus import stopwords

# ============================================================
# SETUP
# ============================================================

OUTPUT_DIR = 'Domain_Stopwords_Analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_PATH = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Preprocessing/dataset_final_sentences_english.csv'

df = pd.read_csv(DATA_PATH)
docs_all = df['sentence'].tolist()
total_docs = len(docs_all)
print(f"Total kalimat: {total_docs:,}")

# Pola token sama dengan yang dipakai di pipeline BERTopic, supaya
# kata yang dianalisis di sini persis kata yang nanti masuk ke vectorizer
TOKEN_PATTERN = re.compile(r'(?u)\b[a-zA-Z]{2,}\b')

# Stopword NLTK dikeluarkan dari analisis: sudah ditangani lapis lain,
# kalau diikutkan hanya akan memenuhi peringkat atas dengan "the/is/and"
stopwords_nltk = set(stopwords.words('english'))

# ============================================================
# TAHAP 1 — HITUNG TERM FREQUENCY
# ============================================================

term_freq = Counter()   # total kemunculan (TF)
doc_freq = Counter()    # jumlah kalimat yang mengandung kata (DF, pendamping)

for doc in docs_all:
    tokens = TOKEN_PATTERN.findall(doc.lower())
    tokens = [t for t in tokens if t not in stopwords_nltk]
    term_freq.update(tokens)
    doc_freq.update(set(tokens))

total_tokens = sum(term_freq.values())

tf_df = pd.DataFrame([
    {
        'rank': 0,
        'kata': w,
        'term_frequency': tf,
        'pct_dari_total_token': round(tf / total_tokens * 100, 4),
        'document_frequency': doc_freq[w],
        'pct_kalimat': round(doc_freq[w] / total_docs * 100, 4),
    }
    for w, tf in term_freq.items()
]).sort_values('term_frequency', ascending=False).reset_index(drop=True)

tf_df['rank'] = tf_df.index + 1
tf_df.to_csv(os.path.join(OUTPUT_DIR, 'tf_seluruh_kata.csv'), index=False)

print(f"\nTotal kata unik    : {len(tf_df):,}")
print(f"Total token         : {total_tokens:,}")
print(f"-> tf_seluruh_kata.csv")

# ============================================================
# TAHAP 2 — AMBIL TOP 20%
# ============================================================

TOP_PERCENT = 0.20
n_top = int(len(tf_df) * TOP_PERCENT)
top_df = tf_df.head(n_top).copy()

# Menghitung variabel tambahan untuk catatan kesimpulan
cakupan_persen = (top_df['term_frequency'].sum() / total_tokens) * 100
sisa_kata = len(tf_df) - n_top
sisa_persen_kosakata = 100 - int(TOP_PERCENT * 100)
sisa_persen_token = 100 - cakupan_persen

print("\n" + "="*70)
print(f"TOP {int(TOP_PERCENT*100)}% KEYWORD BERDASARKAN TF")
print("="*70)
print(f"Jumlah kata yang diambil : {n_top:,} dari {len(tf_df):,} kata unik")
print(f"TF terendah di batas ini : {top_df['term_frequency'].iloc[-1]:,} kemunculan")
print(f"Cakupan token            : {cakupan_persen:.2f}% dari seluruh token di corpus")
print("-" * 70)
print("💡 CATATAN ANALISIS (ARTINYA):")
print(f"> {n_top:,} kata teratas (top {int(TOP_PERCENT*100)}% dari kosakata unik) menyumbang ")
print(f"  {cakupan_persen:.2f}% dari seluruh {total_tokens:,} kemunculan kata di corpus.")
print(f"> Sisanya — {sisa_kata:,} kata ({sisa_persen_kosakata}% kosakata unik) — cuma menyumbang ")
print(f"  {sisa_persen_token:.2f}% token. Itu adalah kata-kata langka yang muncul sekali-dua kali saja.")


kolom_final = ['rank', 'kata', 'term_frequency', 'pct_dari_total_token',
               'document_frequency', 'pct_kalimat']
top_df[kolom_final].to_csv(
    os.path.join(OUTPUT_DIR, 'tf_top20persen_kurasi.csv'), index=False
)
print(f"-> tf_top20persen_kurasi.csv (siap dikurasi manual)")


Total kalimat: 72,374

Total kata unik    : 16,565
Total token         : 466,182
-> tf_seluruh_kata.csv

TOP 20% KEYWORD BERDASARKAN TF
Jumlah kata yang diambil : 3,313 dari 16,565 kata unik
TF terendah di batas ini : 10 kemunculan
Cakupan token            : 93.71% dari seluruh token di corpus
----------------------------------------------------------------------
💡 CATATAN ANALISIS (ARTINYA):
> 3,313 kata teratas (top 20% dari kosakata unik) menyumbang 
  93.71% dari seluruh 466,182 kemunculan kata di corpus.
> Sisanya — 13,252 kata (80% kosakata unik) — cuma menyumbang 
  6.29% token. Itu adalah kata-kata langka yang muncul sekali-dua kali saja.
-> tf_top20persen_kurasi.csv (siap dikurasi manual)


In [2]:
"""
PROMPT CLAUDE : "berikut hasil berupa excel dari top 20% keyword berdasarkan tf, tolong bantu kurasi manual keyword 
apa saja yang cocok untuk dijadikan domain_stopwords untuk keperluan bertopic dan tolong kategorikan biar mudah dianalisa dan diperiksa."
"""

import os
import pandas as pd

OUTPUT_DIR = 'Domain_Stopwords_Analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TF_PATH = os.path.join(OUTPUT_DIR, 'tf_top20persen_kurasi.csv')
TF_FULL_PATH = os.path.join(OUTPUT_DIR, 'tf_seluruh_kata.csv')

KURASI = {
    "1_nama_lokasi": {
        # Fokus penelitian adalah aspek kualitas, bukan identitas lokasi.
        # Nama pantai justru merusak clustering karena mengelompokkan kalimat
        # berdasarkan lokasi, bukan berdasarkan apa yang dibicarakan.
        "bali", "kuta", "nusa", "penida", "pandawa", "sanur", "kelingking",
        "padang", "seminyak", "dua", "uluwatu", "indonesia", "indonesian",
        "balinese", "rex", "south", "international",
    },
    "2_domain_umum": {
        # Muncul di hampir semua kalimat karena memang itu objek penelitian,
        # sehingga tidak membedakan satu topik dari topik lain.
        "beach", "beaches", "place", "places", "island", "tourist", "tourists",
        "tourism", "destination", "destinations", "area", "areas", "spot",
        "spots", "location", "visitors", "world", "life", "holiday", "vacation",
        "trip", "coast", "people", "person", "experience",
    },
    "3_sifat_evaluatif_umum": {
        # Dapat dipakai untuk aspek APA PUN ("beautiful view", "beautiful sand",
        # "beautiful sunset"), sehingga tidak menunjuk aspek tertentu.
        # Polaritas sentimennya ditangani pada tahap analisis sentimen (DeBERTa),
        # bukan pada tahap identifikasi aspek.
        "beautiful", "nice", "good", "great", "best", "perfect", "amazing",
        "stunning", "better", "lovely", "wonderful", "awesome", "fantastic",
        "magnificent", "spectacular", "incredible", "excellent", "fine",
        "super", "special", "unique", "iconic", "famous", "popular",
        "breathtaking", "paradise", "bad", "shame", "ideal", "suitable",
        "comfortable", "pleasant", "happy", "favorite", "worth", "interesting",
        "real", "pretty", "cool", "big", "small", "large", "huge", "main",
        "complete", "god", "wow", "beauty", "natural", "new",
    },
    "4_kata_penguat": {
        # Menguatkan kata lain tanpa membawa makna aspek sendiri.
        "really", "quite", "truly", "absolutely", "definitely", "extremely",
        "incredibly", "simply", "highly", "completely", "relatively",
        "probably", "actually", "maybe", "especially", "totally", "obviously",
        "bit", "well", "much", "very",
    },
    "5_kata_kerja_umum": {
        # Menggambarkan aktivitas berkunjung secara generik, bukan aspek kualitas.
        "go", "going", "get", "getting", "gets", "got", "take", "takes",
        "taking", "took", "see", "seeing", "saw", "seen", "come", "coming",
        "came", "visit", "visited", "visiting", "want", "make", "makes",
        "making", "made", "know", "known", "think", "look", "looking", "looks",
        "find", "found", "try", "trying", "need", "let", "say", "said", "keep",
        "give", "put", "bring", "use", "used", "leave", "left", "stay", "went",
        "arrive", "arrived", "reach", "spend", "sit", "sitting", "watch",
        "watching", "miss", "forget", "offer", "offers", "offering",
        "enter", "wear", "prepared", "able", "seems", "feel", "feels",
        "recommend", "recommended", "enjoy", "enjoyed", "enjoying", "love",
        "loved", "like", "avoid", "play", "playing", "doing",
    },
    "6_waktu_kuantitas_posisi": {
        # Memberi konteks kuantitas/waktu/arah, bukan aspek kualitas.
        "one", "two", "many", "lot", "lots", "time", "times", "day", "days",
        "morning", "afternoon", "evening", "night", "hour", "hours", "minutes",
        "year", "years", "month", "first", "last", "half", "per", "thousand",
        "least", "enough", "plenty", "full", "almost", "less", "every",
        "everything", "everyone", "everywhere", "something", "anything",
        "anyone", "nothing", "part", "whole", "end", "side",
        "front", "behind", "around", "along", "away", "near", "close",
        "next", "nearby", "late", "early", "short", "long", "soon", "already",
        "still", "yet", "always", "never", "ever", "sometimes", "often",
        "since", "due", "though", "however", "although", "otherwise",
        "unfortunately", "overall", "several", "another", "etc", "us", "also",
        "even", "back", "way", "thing", "things", "course", "moment", "may",
        "might", "could", "would", "must", "cannot", "okay", "yes",
    },
}

LABEL_KATEGORI = {
    "1_nama_lokasi": "Nama lokasi & entitas",
    "2_domain_umum": "Kata domain umum",
    "3_sifat_evaluatif_umum": "Kata sifat evaluatif umum",
    "4_kata_penguat": "Kata penguat",
    "5_kata_kerja_umum": "Kata kerja umum",
    "6_waktu_kuantitas_posisi": "Penanda waktu/jumlah/posisi",
}

# ============================================================
# GABUNGKAN DENGAN STATISTIK TF
# ============================================================

tf_df = pd.read_csv(TF_PATH)
# Sebagian kata mungkin ada di daftar penuh tapi di luar top 20%;
# file lengkap dipakai sebagai cadangan pencarian statistik.
if os.path.exists(TF_FULL_PATH):
    tf_full = pd.read_csv(TF_FULL_PATH)
else:
    tf_full = tf_df

stat_lookup = tf_full.set_index('kata')[
    ['rank', 'term_frequency', 'pct_dari_total_token',
     'document_frequency', 'pct_kalimat']
].to_dict('index')

baris = []
tanpa_statistik = []

for kode_kategori, daftar_kata in KURASI.items():
    for kata in daftar_kata:
        stat = stat_lookup.get(kata)
        if stat is None:
            # Tidak muncul di analisis TF -- umumnya karena kata tersebut
            # sudah termasuk stopword NLTK sehingga dikecualikan sejak awal
            tanpa_statistik.append((kata, kode_kategori))
            continue
        baris.append({
            'keyword': kata,
            'kategori': LABEL_KATEGORI[kode_kategori],
            'kode_kategori': kode_kategori,
            'rank': stat['rank'],
            'term_frequency': stat['term_frequency'],
            'pct_dari_total_token': stat['pct_dari_total_token'],
            'document_frequency': stat['document_frequency'],
            'pct_kalimat': stat['pct_kalimat'],
        })

hasil = pd.DataFrame(baris)

hasil = hasil.sort_values(
    ['kode_kategori', 'term_frequency'], ascending=[True, False]
).reset_index(drop=True)

kolom_final = ['keyword', 'kategori', 'rank', 'term_frequency',
               'pct_dari_total_token', 'document_frequency', 'pct_kalimat']
hasil[kolom_final].to_csv(
    os.path.join(OUTPUT_DIR, 'domain_stopwords_final.csv'), index=False
)

# ============================================================
# REKAP PER KATEGORI
# ============================================================

rekap = (hasil.groupby(['kode_kategori', 'kategori'])
         .agg(jumlah_kata=('keyword', 'count'),
              total_term_frequency=('term_frequency', 'sum'),
              pct_token_disaring=('pct_dari_total_token', 'sum'))
         .reset_index()
         .sort_values('kode_kategori'))
rekap['pct_token_disaring'] = rekap['pct_token_disaring'].round(2)
rekap[['kategori', 'jumlah_kata', 'total_term_frequency', 'pct_token_disaring']].to_csv(
    os.path.join(OUTPUT_DIR, 'domain_stopwords_rekap.csv'), index=False
)

# ============================================================
# LAPORAN
# ============================================================

print("=" * 78)
print("REKAP PER KATEGORI")
print("=" * 78)
print(rekap[['kategori', 'jumlah_kata', 'total_term_frequency',
             'pct_token_disaring']].to_string(index=False))
print("-" * 78)
print(f"{'TOTAL':<30} {len(hasil):>6} {hasil['term_frequency'].sum():>14,} "
      f"{hasil['pct_dari_total_token'].sum():>8.2f}")

if tanpa_statistik:
    print("\n" + "=" * 78)
    print("CATATAN — kata tanpa statistik TF")
    print("=" * 78)
    print("Kata berikut ada di daftar kurasi namun tidak muncul pada analisis TF,")
    print("umumnya karena sudah termasuk stopword NLTK (dikecualikan sejak awal),")
    print("sehingga memasukkannya ke domain_stopwords bersifat redundan:")
    for kata, kat in sorted(tanpa_statistik):
        print(f"   - {kata:<15} ({LABEL_KATEGORI[kat]})")

print("\nFile tersimpan:")
print(f"  {OUTPUT_DIR}/domain_stopwords_final.csv")
print(f"  {OUTPUT_DIR}/domain_stopwords_rekap.csv")


REKAP PER KATEGORI
                   kategori  jumlah_kata  total_term_frequency  pct_token_disaring
      Nama lokasi & entitas           17                 14623                3.14
           Kata domain umum           25                 48299               10.36
  Kata sifat evaluatif umum           50                 40431                8.67
               Kata penguat           20                 12616                2.71
            Kata kerja umum           80                 44598                9.57
Penanda waktu/jumlah/posisi           92                 54115               11.61
------------------------------------------------------------------------------
TOTAL                             284        214,682    46.05

CATATAN — kata tanpa statistik TF
Kata berikut ada di daftar kurasi namun tidak muncul pada analisis TF,
umumnya karena sudah termasuk stopword NLTK (dikecualikan sejak awal),
sehingga memasukkannya ke domain_stopwords bersifat redundan:
   - doing          

## Kandidat Aspek

In [3]:
# ============================================================================
# DAFTAR ASPEK KANDIDAT (SEED) — kata ber-TF tinggi yang TIDAK dijadikan stopword
# ============================================================================
"""
PROMPT CLAUDE : "berikut hasil berupa excel dari top 20% keyword berdasarkan tf, tolong bantu kurasi manual keyword 
apa saja yang cocok untuk dijadikan kandidat aspek untuk keperluan bertopic dan tolong kategorikan biar mudah dianalisa dan diperiksa."
"""

import os
import pandas as pd

OUTPUT_DIR = 'Domain_Stopwords_Analysis'
TF_PATH = os.path.join(OUTPUT_DIR, 'tf_top20persen_kurasi.csv')
TF_FULL_PATH = os.path.join(OUTPUT_DIR, 'tf_seluruh_kata.csv')

ASPEK_KANDIDAT = {
    'kebersihan': [
        'clean', 'dirty', 'trash', 'garbage', 'rubbish', 'waste', 'plastic',
        'seaweed', 'maintained', 'pristine', 'clear', 'crystal', 'cleanliness',
        'cleaner', 'litter', 'cleaning', 'cleaned', 'cleanest', 'cleans',
        'cleaners', 'littered', 'littering', 'debris', 'bottles', 'bottle',
        'plastics', 'polluted', 'pollution', 'sewage', 'filthy', 'filth', 'dirt',
        'dirtiest', 'dirtier', 'smell', 'smells', 'smelly', 'smelled', 'dump',
        'dumped', 'bins', 'bin', 'tidy', 'wrappers', 'cigarette', 'butts', 'straws',
        'junk', 'hygiene', 'spotless', 'trashes', 'murky', 'algae', 'seaweeds',
        'mess', 'messy', 'maintenance', 'maintain', 'maintaining',
    ],
    'kepadatan': [
        'crowded', 'crowds', 'crowd', 'busy', 'packed', 'overcrowded', 'quiet',
        'peaceful', 'calm', 'queue', 'touristy', 'wait', 'waiting', 'line',
        'quieter', 'queues', 'queuing', 'lines', 'waited', 'busier', 'busiest',
        'congested', 'crowdy', 'crowed', 'cramped', 'crammed', 'overrun',
        'deserted', 'empty', 'uncrowded', 'secluded', 'hordes', 'masses', 'noisy',
        'noise', 'loud', 'tranquil', 'tranquility', 'serene', 'serenity', 'peace',
        'peacefully', 'solitude', 'bustling', 'bustle', 'hustle', 'hectic',
        'chaotic', 'chaos',
    ],
    'aksesibilitas': [
        'access', 'stairs', 'steps', 'steep', 'climb', 'climbing', 'descent',
        'hike', 'trek', 'path', 'road', 'roads', 'parking', 'narrow', 'elevator',
        'construction', 'building', 'tiring', 'difficult', 'challenging', 'effort',
        'walk', 'walking', 'hill', 'shoes', 'easy', 'easily', 'far', 'top',
        'bottom', 'hard', 'accessible', 'descend', 'trail', 'hiking', 'distance',
        'drive', 'driver', 'staircase', 'stairway', 'stair', 'step', 'ascent',
        'ascend', 'descending', 'climbed', 'hiked', 'hikers', 'hiker', 'trekking',
        'footwear', 'sneakers', 'sandals', 'slippers', 'trainers', 'boots',
        'flipflops', 'flip', 'flops', 'slippery', 'slipping', 'slip', 'uneven',
        'bumpy', 'potholes', 'paved', 'unpaved', 'paths', 'pathway', 'walkway',
        'footpath', 'footing', 'railings', 'railing', 'handrails', 'rails', 'ropes',
        'rope', 'slope', 'slopes', 'sloping', 'steeper', 'uphill', 'downhill',
        'winding', 'vertical', 'strenuous', 'exhausting', 'arduous', 'tiresome',
        'tricky', 'scramble', 'scrambling', 'terrain', 'route', 'track', 'lift',
        'taxi', 'taxis', 'transportation', 'transport', 'shuttle', 'bus', 'buses',
        'gojek', 'vehicle', 'vehicles', 'parked', 'drivers', 'driving', 'jam',
        'jams', 'highway', 'toll', 'accessibility', 'accessed', 'mobility',
        'stamina', 'fitness', 'physically', 'knee', 'knees', 'halfway', 'easier',
        'easiest', 'harder', 'tough', 'doable', 'manageable', 'demanding',
        'difficulty', 'difficulties', 'challenge', 'challenges', 'walked', 'walks',
        'walkers', 'pedestrian', 'pedestrians', 'sidewalk', 'sidewalks',
        'boardwalk', 'promenade',
    ],
    'harga': [
        'price', 'prices', 'fee', 'free', 'expensive', 'cheap', 'affordable',
        'costs', 'money', 'pay', 'paid', 'entrance', 'entry', 'ticket', 'charge',
        'rupiah', 'idr', 'rp', 'rupees', 'buy', 'sell', 'selling', 'cost',
        'tickets', 'fees', 'paying', 'charged', 'charging', 'charges', 'admission',
        'cheaper', 'cheapest', 'overpriced', 'pricey', 'priced', 'inexpensive',
        'exorbitant', 'bargain', 'bargaining', 'haggle', 'haggling', 'negotiate',
        'cash', 'payment', 'budget', 'rubles', 'usd', 'dollars', 'dollar', 'aud',
        'rate', 'rates', 'reasonable', 'reasonably', 'tax', 'purchase', 'buying',
        'bought', 'sold', 'sells',
    ],
    'fasilitas': [
        'toilet', 'toilets', 'facilities', 'umbrella', 'umbrellas', 'chairs',
        'loungers', 'shade', 'rent', 'rentals', 'public', 'service', 'available',
        'sunbeds', 'shower', 'showers', 'chair', 'rental', 'services', 'staff',
        'restrooms', 'restroom', 'bathrooms', 'bathroom', 'sunbed', 'lounger',
        'lounge', 'lounges', 'parasols', 'parasol', 'seating', 'seats', 'seat',
        'tables', 'table', 'beanbags', 'beanbag', 'towel', 'towels', 'amenities',
        'infrastructure', 'facility', 'wash', 'washing', 'rinse', 'rinsing',
        'shaded', 'shady', 'shelter', 'tent', 'tents', 'huts', 'internet',
        'attendant', 'renting', 'rented', 'hire', 'hired', 'hiring', 'equipment',
    ],
    'kuliner': [
        'food', 'drinks', 'drink', 'restaurants', 'cafes', 'bars', 'snacks', 'beer',
        'coconut', 'eat', 'restaurant', 'cafe', 'bar', 'delicious', 'warungs',
        'coffee', 'breakfast', 'lunch', 'dinner', 'meal', 'meals', 'seafood',
        'eateries', 'dining', 'cuisine', 'culinary', 'dishes', 'menu', 'snack',
        'corn', 'grilled', 'fried', 'roasted', 'noodles', 'rice', 'satay',
        'meatballs', 'nasi', 'goreng', 'chicken', 'cocktails', 'cocktail', 'beers',
        'bintang', 'bintangs', 'juice', 'coconuts', 'beverage', 'beverages',
        'refreshments', 'drinking', 'eating', 'ate', 'warung', 'resto', 'tasty',
        'taste', 'tastes', 'fruit', 'fruits', 'tea', 'alcohol', 'pubs', 'foods',
        'egg', 'soup', 'chips', 'sauce', 'potato',
    ],
    'pemandangan': [
        'view', 'views', 'sunset', 'sunsets', 'sunrise', 'scenery', 'viewpoint',
        'picture', 'pictures', 'photo', 'photos', 'instagram', 'scenic', 'sky',
        'landscape', 'landscapes', 'panorama', 'panoramic', 'picturesque',
        'sunrises', 'horizon', 'backdrop', 'viewing', 'viewpoints', 'lookout',
        'overlooking', 'photography', 'photographer', 'photographers', 'photogenic',
        'photoshoot', 'selfie', 'selfies', 'pics', 'pic', 'instagrammable',
        'instagramable', 'insta', 'camera', 'drone', 'sight', 'sights', 'postcard',
        'capture', 'capturing', 'skies',
    ],
    'air_ombak': [
        'water', 'waves', 'wave', 'sea', 'ocean', 'surf', 'surfing', 'surfers',
        'swim', 'swimming', 'tide', 'current', 'shallow', 'blue', 'turquoise',
        'rough', 'strong', 'waters', 'currents', 'reef', 'deep', 'tides', 'tidal',
        'seawater', 'swimmers', 'swimmer', 'swam', 'swimmable', 'bathing', 'bathe',
        'dip', 'surfer', 'surfboard', 'surfboards', 'surfed', 'rip', 'rips',
        'swell', 'swells', 'choppy', 'wavy', 'crashing', 'deeper', 'depth', 'seas',
        'lagoon', 'azure', 'emerald', 'salty', 'underwater',
    ],
    'fisik_pantai': [
        'sand', 'sandy', 'white', 'soft', 'rocks', 'rocky', 'rock', 'coral',
        'cliff', 'cliffs', 'spacious', 'wide', 'shore', 'limestone', 'cave',
        'shoreline', 'coastline', 'trees', 'sands', 'stones', 'stone', 'pebbles',
        'boulders', 'shells', 'corals', 'reefs', 'formation', 'formations', 'caves',
        'cove', 'bay', 'coastal', 'seabed', 'cliffside', 'shores', 'powdery',
        'coarse', 'volcanic', 'erosion', 'eroded', 'abrasion', 'breakwater',
        'expanse', 'expansive', 'stretch', 'stretches', 'rugged', 'tree', 'palm',
        'greenery',
    ],
    'keamanan': [
        'safe', 'dangerous', 'careful', 'steal', 'belongings', 'beware', 'safety',
        'unsafe', 'safely', 'safer', 'lifeguards', 'lifeguard', 'security', 'guard',
        'guards', 'danger', 'risk', 'risks', 'risky', 'caution', 'cautious',
        'warning', 'warnings', 'warned', 'warn', 'flags', 'flag', 'drowned',
        'accident', 'accidents', 'injured', 'injuries', 'stolen', 'stealing',
        'stole', 'snatch', 'thieving', 'valuables', 'police', 'rescue', 'scary',
        'treacherous', 'vertigo', 'unattended',
    ],
    'satwa': [
        'monkeys', 'dogs', 'monkey', 'dog', 'stray', 'animals', 'turtle', 'turtles',
        'fish', 'jellyfish', 'urchins', 'manta', 'mantas', 'starfish', 'crabs',
        'bird', 'rats',
    ],
    'pedagang': [
        'vendors', 'sellers', 'stalls', 'shops', 'locals', 'mall', 'shopping',
        'hawkers', 'seller', 'vendor', 'traders', 'stall', 'stores', 'store',
        'shop', 'market', 'markets', 'kiosks', 'kiosk', 'souvenirs', 'souvenir',
        'sarongs', 'sarong', 'bracelets', 'trinkets', 'wares', 'goods',
        'merchandise', 'pushy', 'hassle', 'hassled', 'hassling', 'harassed',
        'harassing', 'persistent', 'touts', 'boutiques', 'malls', 'beachwalk',
        'braiding', 'braids',
    ],
    'aktivitas': [
        'activities', 'jogging', 'sports', 'boat', 'massage', 'ride', 'scooter',
        'motorbike', 'bike', 'car', 'cars', 'traffic', 'music', 'chill', 'vibe',
        'atmosphere', 'relaxing', 'relax', 'fun', 'lively', 'sunbathing', 'tour',
        'guide', 'lessons', 'canoe', 'boats', 'board', 'beginners', 'stroll',
        'explore', 'activity', 'canoeing', 'canoes', 'kayaking', 'kayak', 'paddle',
        'paragliding', 'parasailing', 'snorkeling', 'snorkelling', 'snorkel',
        'diving', 'dive', 'divers', 'scuba', 'watersports', 'sport', 'jet', 'ski',
        'skis', 'banana', 'kite', 'kites', 'volleyball', 'football', 'swing',
        'cycling', 'cycle', 'cyclists', 'bicycle', 'bicycles', 'bikes', 'scooters',
        'motorbikes', 'motorcycle', 'motorcycles', 'jog', 'yoga', 'fishing',
        'picnic', 'sunbathe', 'sunbath', 'sunbathers', 'tanning', 'lounging',
        'relaxed', 'relaxation', 'unwind', 'chilling', 'chilled', 'leisurely',
        'leisure', 'recreation', 'recreational', 'entertainment', 'nightlife',
        'party', 'parties', 'clubs', 'club', 'dance', 'kecak', 'performance',
        'performances', 'tours', 'guides', 'instructors', 'instructor', 'lesson',
        'beginner', 'intermediate', 'learn', 'learning', 'massages', 'spa', 'spas',
        'vibes', 'ambience', 'ambiance', 'vibrant', 'strolling', 'exploring',
        'explored', 'sightseeing', 'hangout',
    ],
    'pengunjung': [
        'children', 'kids', 'family', 'families', 'friends', 'friendly', 'child',
        'kid', 'baby', 'adults', 'adult', 'elderly', 'couples', 'parents',
        'foreigners', 'foreigner', 'domestic', 'travelers', 'travellers',
        'backpackers', 'influencers', 'instagrammers', 'westerners',
    ],
    'cuaca': [
        'weather', 'hot', 'sun', 'cold', 'heat', 'season', 'warm', 'breeze',
        'sunny', 'rainy', 'rain', 'raining', 'rained', 'rains', 'windy', 'wind',
        'winds', 'cloudy', 'clouds', 'humid', 'temperature', 'scorching', 'blazing',
        'cooler', 'storm', 'sunlight', 'rays', 'sunscreen', 'sunblock', 'hat',
        'hats', 'summer', 'degrees',
    ],
    'akomodasi': [
        'resort', 'resorts', 'room', 'stayed', 'staying', 'hotel', 'hotels',
        'accommodation', 'accommodations', 'villa', 'villas', 'rooms', 'hyatt',
        'westin', 'sofitel', 'mercure', 'overnight',
    ],
    'lain': [
        'pandemic', 'hidden', 'covid', 'corona', 'coronavirus', 'ppkm',
    ],
}

tf_top = pd.read_csv(TF_PATH)
tf_full = pd.read_csv(TF_FULL_PATH) if os.path.exists(TF_FULL_PATH) else tf_top
kol_stat = ['rank', 'term_frequency', 'pct_dari_total_token', 'document_frequency', 'pct_kalimat']

seed = pd.DataFrame([{'aspek': a, 'keyword': k}
                     for a, daftar in ASPEK_KANDIDAT.items() for k in daftar])

# --- Pemeriksaan konsistensi ---
stop_domain = set(kata for daftar in KURASI.values() for kata in daftar)
dobel = seed[seed.duplicated('keyword', keep=False)].sort_values('keyword')
assert dobel.empty, f"Kata masuk lebih dari satu aspek:\n{dobel}"
bentrok = sorted(set(seed['keyword']) & stop_domain)
assert not bentrok, f"Kata ada di aspek DAN di domain stopword: {bentrok}"

seed = seed.merge(tf_full[['kata'] + kol_stat].rename(columns={'kata': 'keyword'}),
                  on='keyword', how='left')
seed['di_top20_tf'] = seed['keyword'].isin(set(tf_top['kata']))
seed = seed.sort_values(['aspek', 'term_frequency'], ascending=[True, False])
seed = seed.set_index('aspek').loc[list(ASPEK_KANDIDAT)].reset_index()   # urutan aspek tetap
seed.to_csv(os.path.join(OUTPUT_DIR, 'aspek_kandidat_seed.csv'), index=False)

rekap_aspek = (seed.groupby('aspek', sort=False)
               .agg(jumlah_kata=('keyword', 'count'),
                    total_term_frequency=('term_frequency', 'sum'),
                    pct_token=('pct_dari_total_token', 'sum')).reset_index())
rekap_aspek['pct_token'] = rekap_aspek['pct_token'].round(2)
rekap_aspek.to_csv(os.path.join(OUTPUT_DIR, 'aspek_kandidat_rekap.csv'), index=False)

print("=" * 78 + "\nREKAP ASPEK KANDIDAT\n" + "=" * 78)
print(rekap_aspek.to_string(index=False))

tanpa_stat = seed.loc[seed['term_frequency'].isna(), 'keyword'].tolist()
luar_top = seed.loc[~seed['di_top20_tf'] & seed['term_frequency'].notna(), 'keyword'].tolist()
if tanpa_stat:
    print(f"\nKata seed yang TIDAK ada di korpus (periksa ejaan): {tanpa_stat}")
if luar_top:
    print(f"Kata seed di luar top 20% TF: {luar_top}")

# --- Seberapa besar bagian top-20% TF yang sudah dikategorikan? ---
top = tf_top.copy()
top['status'] = 'belum dikategorikan'
top.loc[top['kata'].isin(stop_domain), 'status'] = 'domain stopword'
top.loc[top['kata'].isin(set(seed['keyword'])), 'status'] = 'seed aspek'
cakupan = top.groupby('status').agg(jumlah_kata=('kata', 'count'),
                                    pct_token=('pct_dari_total_token', 'sum')).round(2)
print("\n" + "=" * 78 + "\nPEMBAGIAN KATA TOP 20% TF\n" + "=" * 78)
print(cakupan.to_string())
print("\n40 kata ber-TF tertinggi yang BELUM dikategorikan (tinjau: stopword, aspek, atau biarkan?):")
print(top[top['status'] == 'belum dikategorikan'].head(40)[
    ['rank', 'kata', 'term_frequency', 'pct_kalimat']].to_string(index=False))
top[top['status'] == 'belum dikategorikan'].to_csv(
    os.path.join(OUTPUT_DIR, 'tf_top20persen_belum_dikategorikan.csv'), index=False)

print(f"\nFile tersimpan:\n  {OUTPUT_DIR}/aspek_kandidat_seed.csv\n  {OUTPUT_DIR}/aspek_kandidat_rekap.csv"
      f"\n  {OUTPUT_DIR}/tf_top20persen_belum_dikategorikan.csv")

REKAP ASPEK KANDIDAT
        aspek  jumlah_kata  total_term_frequency  pct_token
   kebersihan           59                 10369       2.22
    kepadatan           49                  8599       1.84
aksesibilitas          143                 21711       4.66
        harga           60                 10884       2.33
    fasilitas           60                  5991       1.28
      kuliner           69                  7738       1.66
  pemandangan           46                 13359       2.87
    air_ombak           50                 17452       3.74
 fisik_pantai           50                  9115       1.96
     keamanan           44                  3190       0.68
        satwa           17                  1872       0.40
     pedagang           41                  4001       0.86
    aktivitas          115                 11474       2.46
   pengunjung           23                  3006       0.64
        cuaca           33                  3759       0.81
    akomodasi      